# Credit Scoring — Exploratory Data Analysis

**Objectif** : Comprendre les données, identifier les variables importantes, et préparer le feature engineering pour prédire la probabilité de défaut de paiement (`TARGET = 1`).

**Dataset** : Home Credit Default Risk (Kaggle)

---

## 0. Imports et configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

%matplotlib inline

print('Imports OK')

## 1. Chargement des données

In [ ]:
DATA_PATH = '../../data/'

# Table principale
train = pd.read_csv(DATA_PATH + 'application_train.csv')
test  = pd.read_csv(DATA_PATH + 'application_test.csv')

print(f'Train : {train.shape[0]:,} lignes, {train.shape[1]} colonnes')
print(f'Test  : {test.shape[0]:,} lignes, {test.shape[1]} colonnes')

In [ ]:
train.head(3)

## 2. Distribution de la variable cible (TARGET)

In [ ]:
target_counts = train['TARGET'].value_counts()
target_pct    = train['TARGET'].value_counts(normalize=True) * 100

print('Distribution TARGET :')
print(f'  0 (remboursé)  : {target_counts[0]:,} ({target_pct[0]:.1f}%)')
print(f'  1 (défaut)     : {target_counts[1]:,} ({target_pct[1]:.1f}%)')
print(f'\n=> Déséquilibre classes : {target_pct[0]:.1f}% vs {target_pct[1]:.1f}%')
print('=> Accuracy inutile ici — on utilisera AUC-ROC')

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(['Remboursé (0)', 'Défaut (1)'], target_counts.values, color=['#2ecc71', '#e74c3c'])
ax.set_title('Distribution de la variable TARGET')
ax.set_ylabel('Nombre de clients')
for i, v in enumerate(target_counts.values):
    ax.text(i, v + 500, f'{v:,}\n({target_pct.iloc[i]:.1f}%)', ha='center')
plt.tight_layout()
plt.show()

## 3. Analyse des valeurs manquantes

In [ ]:
missing = train.isnull().sum()
missing_pct = (missing / len(train) * 100).round(2)
missing_df = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
missing_df = missing_df[missing_df['missing_count'] > 0].sort_values('missing_pct', ascending=False)

print(f'Colonnes avec valeurs manquantes : {len(missing_df)} / {train.shape[1]}')
print(f'Colonnes avec >50% manquants : {(missing_df["missing_pct"] > 50).sum()}')
print('\nTop 20 colonnes avec le plus de manquants :')
missing_df.head(20)

In [ ]:
# Visualisation des 30 colonnes avec le plus de manquants
top_missing = missing_df.head(30)
fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(top_missing.index, top_missing['missing_pct'], color='#e67e22')
ax.axvline(x=50, color='red', linestyle='--', label='50% seuil')
ax.set_xlabel('% valeurs manquantes')
ax.set_title('Top 30 colonnes avec le plus de valeurs manquantes')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Types de variables

In [ ]:
num_cols = train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = train.select_dtypes(include=['object']).columns.tolist()

# Retirer TARGET de num_cols
num_cols = [c for c in num_cols if c != 'TARGET']

print(f'Variables numériques : {len(num_cols)}')
print(f'Variables catégorielles : {len(cat_cols)}')
print(f'\nVariables catégorielles : {cat_cols}')

## 5. Variables numériques clés

In [ ]:
# Variables les plus importantes a priori
key_num = ['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE',
           'DAYS_BIRTH', 'DAYS_EMPLOYED', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']

# Convertir DAYS_BIRTH en années (valeur négative = jours depuis naissance)
train['AGE_YEARS'] = -train['DAYS_BIRTH'] / 365

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

for i, col in enumerate(key_num):
    if col in train.columns:
        train[train['TARGET'] == 0][col].dropna().plot.hist(
            ax=axes[i], bins=50, alpha=0.6, color='#2ecc71', label='Remboursé'
        )
        train[train['TARGET'] == 1][col].dropna().plot.hist(
            ax=axes[i], bins=50, alpha=0.6, color='#e74c3c', label='Défaut'
        )
        axes[i].set_title(col)
        axes[i].legend(fontsize=8)

plt.suptitle('Distribution des variables clés par TARGET', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

## 6. Variables catégorielles

In [ ]:
# Taux de défaut par variable catégorielle
for col in cat_cols:
    default_rate = train.groupby(col)['TARGET'].mean().sort_values(ascending=False)
    print(f'\n--- {col} ---')
    print(default_rate.to_string())

In [ ]:
# Visualisation taux de défaut par type de contrat et genre
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, col in zip(axes, ['NAME_CONTRACT_TYPE', 'CODE_GENDER']):
    rates = train.groupby(col)['TARGET'].mean().sort_values(ascending=False)
    rates.plot.bar(ax=ax, color='#3498db', edgecolor='black')
    ax.set_title(f'Taux de défaut par {col}')
    ax.set_ylabel('Taux de défaut')
    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

## 7. EXT_SOURCE — Variables les plus prédictives

In [ ]:
# EXT_SOURCE_1/2/3 sont des scores externes très prédictifs
ext_sources = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']

for col in ext_sources:
    corr = train[col].corr(train['TARGET'])
    print(f'{col} — corrélation avec TARGET : {corr:.4f}')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ext_sources):
    train.boxplot(column=col, by='TARGET', ax=ax)
    ax.set_title(col)
    ax.set_xlabel('TARGET (0=OK, 1=Défaut)')

plt.suptitle('EXT_SOURCE par TARGET')
plt.tight_layout()
plt.show()

## 8. Corrélation avec TARGET — Top variables

In [ ]:
# Corrélation de toutes les variables numériques avec TARGET
correlations = train[num_cols + ['TARGET']].corr()['TARGET'].drop('TARGET')
correlations = correlations.abs().sort_values(ascending=False)

print('Top 20 variables corrélées avec TARGET (valeur absolue) :')
print(correlations.head(20).to_string())

fig, ax = plt.subplots(figsize=(10, 6))
correlations.head(20).plot.barh(ax=ax, color='#9b59b6')
ax.set_title('Top 20 corrélations |r| avec TARGET')
ax.set_xlabel('|Corrélation de Pearson|')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 9. Analyse des anomalies

In [ ]:
# DAYS_EMPLOYED : valeur 365243 = anomalie connue (retraités/sans emploi)
print('Valeurs de DAYS_EMPLOYED :')
print(train['DAYS_EMPLOYED'].describe())
print(f"\nNombre de valeurs 365243 (anomalie) : {(train['DAYS_EMPLOYED'] == 365243).sum():,}")

# Taux de défaut avec/sans anomalie
anomaly_rate = train[train['DAYS_EMPLOYED'] == 365243]['TARGET'].mean()
normal_rate  = train[train['DAYS_EMPLOYED'] != 365243]['TARGET'].mean()
print(f'\nTaux défaut avec anomalie  : {anomaly_rate:.3f}')
print(f'Taux défaut sans anomalie  : {normal_rate:.3f}')
print('=> Ces valeurs 365243 portent une info — créer un flag binaire')

## 10. Conclusions EDA — Features à créer

In [ ]:
print("""
=== CONCLUSIONS EDA ===

1. DÉSÉQUILIBRE : ~92% remboursé vs ~8% défaut
   → Utiliser AUC-ROC (pas accuracy), class_weight='balanced'

2. VARIABLES PRÉDICTIVES FORTES :
   - EXT_SOURCE_1/2/3 (scores externes)
   - DAYS_BIRTH (âge)
   - AMT_CREDIT, AMT_ANNUITY, AMT_GOODS_PRICE
   - DAYS_EMPLOYED (avec flag anomalie)

3. FEATURES À CRÉER :
   - CREDIT_INCOME_RATIO = AMT_CREDIT / AMT_INCOME_TOTAL
   - ANNUITY_INCOME_RATIO = AMT_ANNUITY / AMT_INCOME_TOTAL
   - AGE_YEARS = -DAYS_BIRTH / 365
   - DAYS_EMPLOYED_ANOM = flag si DAYS_EMPLOYED == 365243
   - EXT_SOURCE_MEAN = moyenne des 3 EXT_SOURCE

4. VALEURS MANQUANTES :
   - Colonnes >50% manquants : à supprimer ou imputer avec médiane/mode
   - EXT_SOURCE_1 a ~56% manquants → imputation importante
""")